# Advanced Extraction Methods

The examples here build on the basics in `ExtractionExamples.ipynb`. They cover time windows, sensor constraints, intersections of the two, and multi-rule specs. Each section shows a YAML spec and the equivalent Python call using `spec_from_dict`; the two forms are interchangeable.

Run the setup cell first.

In [ ]:
from pathlib import Path

video_dir  = Path("video/")       # folder containing your .mp4 or .mov files
csv_file   = Path("sensors.csv")  # sensor CSV (required for constraint examples)
output_dir = Path("frames/")
output_dir.mkdir(exist_ok=True)

## 1. Filtering by Time Window

Add a `periods` list to a rule to restrict extraction to one or more UTC windows. Outside those windows no frames are taken:

```yaml
# spec.yaml
rules:
  # 1 frame every 2 seconds, but only during this UTC window
  - interval_s: 2.0
    periods:
      - start: "2025-10-15T10:10:00Z"
        end:   "2025-10-15T10:18:00Z"
```

You can also list multiple windows on the same rule. Frames are extracted during *either* window. In this example we are passing a spec from a dictionary and not a YAML file. The syntax is more complicated, but it is another valid use of the library functions that doesn't require a spec file to be written:

In [ ]:
spec = spec_from_dict({
    "rules": [
        {
            "interval_s": 2.0,
            "periods": [
                {"start": "2025-10-15T10:10:00Z", "end": "2025-10-15T10:10:16Z"},
                {"start": "2025-10-15T10:10:36Z", "end": "2025-10-15T10:10:40Z"},
            ],
        },
    ],
})

extract(spec=spec, video_source=video_dir, output_dir=output_dir)

frames = sorted(output_dir.glob("*.jpg"))
print(f"{len(frames)} frames written")

## 2. Filtering by Sensor Condition

Add a `constraints` list to restrict extraction to times when a sensor value falls within a range. Requires a sensor CSV and a `mappings` block:

```yaml
# spec.yaml
rules:
  # 1 frame every 5 seconds, only while depth is between 1000 and 1200 m
  - interval_s: 5.0
    constraints:
      - column: depth
        min: 375
        max: 378

mappings:
  timestamp: Timestamp
  depth:     Depth_m
```

Omit `min` or `max` to leave that end of the range open. Again here we show the spec-from-dictionary input style instead of the YAML:

In [ ]:
from soi_frame_extractor import extract, spec_from_dict

spec = spec_from_dict({
    "rules": [
        {
            "interval_s": 2.0,
            "constraints": [
                {"column": "depth", "min": 377},
            ],
        },
    ],
    "mappings": {
        "timestamp": "Timestamp",
        "depth":     "Depth_m",
    },
})

extract(spec=spec, video_source=video_dir, output_dir=output_dir, csv_path=csv_file)

frames = sorted(output_dir.glob("*.jpg"))
print(f"{len(frames)} frames written")

## 3. Intersection: Time Window AND Sensor Condition

When `periods` and `constraints` are on the **same rule**, they intersect: frames are only extracted when *both* conditions hold at once.

```yaml
# spec.yaml
rules:
  # 1 frame every 2 seconds, only during the UTC window AND while below 500 m
  - interval_s: 2.0
    periods:
      - start: "2025-10-15T10:10:00Z"
        end:   "2025-10-15T10:10:30Z"
    constraints:
      - column: depth
        max: 377.0

mappings:
  timestamp: Timestamp
  depth:     Depth_m
```

In [ ]:
from soi_frame_extractor import extract, spec_from_dict

spec = spec_from_dict({
    "rules": [
        {
            "interval_s": 2.0,
            "periods": [
                {"start": "2025-10-15T10:10:00Z", "end": "2025-10-15T10:10:40Z"},
            ],
            "constraints": [
                {"column": "depth", "max": 377.0},
            ],
        },
    ],
    "mappings": {
        "timestamp": "Timestamp",
        "depth":     "Depth_m",
    },
})

extract(spec=spec, video_source=video_dir, output_dir=output_dir, csv_path=csv_file)

frames = sorted(output_dir.glob("*.jpg"))
print(f"{len(frames)} frames written")

## 4. Multiple Rules

Separate rules contribute independently; their timestamps are merged and deduplicated. Use this when you want different sampling densities under different conditions.

```yaml
# spec.yaml
rules:
  - interval_s: 30.0                  # coarse coverage for the whole dive

  - interval_s: 5.0                   # denser in the 500-1000 m band
    constraints:
      - column: depth
        min: 500
        max: 1000

  - interval_s: 2.0                   # dense during a specific event
    periods:
      - start: "2025-10-15T10:20:00Z"
        end:   "2025-10-15T10:28:00Z"

mappings:
  timestamp: Timestamp
  depth:     Depth_m
```

If two rules land on the same timestamp, only one frame is written.

In [ ]:
from soi_frame_extractor import extract, spec_from_dict

spec = spec_from_dict({
    "rules": [
        # Rule 1: 1 frame every 30 s for the entire session
        {"interval_s": 30.0},

        # Rule 2: 1 frame every 5 s while between 500 and 1000 m
        {
            "interval_s": 5.0,
            "constraints": [
                {"column": "depth", "min": 500, "max": 1000},
            ],
        },

        # Rule 3: 1 frame every 2 s during a specific event
        {
            "interval_s": 2.0,
            "periods": [
                {"start": "2025-10-15T10:20:00Z", "end": "2025-10-15T10:28:00Z"},
            ],
        },
    ],
    "mappings": {
        "timestamp": "Timestamp",
        "depth":     "Depth_m",
    },
})

extract(spec=spec, video_source=video_dir, output_dir=output_dir, csv_path=csv_file)

frames = sorted(output_dir.glob("*.jpg"))
print(f"{len(frames)} frames written")

## 5. Plan and Inspect

You can call the planner directly to see which frames would be extracted (per-video counts, UTC timestamps, and offsets) without decoding any video. Useful for sanity-checking a spec before committing to a full run.

In [ ]:
from pathlib import Path
from datetime import timedelta
from soi_frame_extractor import (
    spec_from_dict, discover_videos, create_video_session,
    create_session_db, import_csv, plan, close_session_db,
)

spec = spec_from_dict({
    "rules": [
        {"interval_s": 30.0},
        {
            "interval_s": 5.0,
            "constraints": [{"column": "depth", "min": 500, "max": 1000}],
        },
    ],
    "mappings": {"timestamp": "Timestamp", "depth": "Depth_m"},
})

session = create_video_session(discover_videos(video_dir))
conn    = create_session_db()
import_csv(csv_file, conn, spec.mappings)
plans   = plan(spec, session, conn)
close_session_db(conn)

total = 0
for p in plans:
    video_start = p.video_file.utc_start
    print(f"{p.video_file.path.name}: {len(p.frames)} frames")
    for f in p.frames:
        utc_time = video_start + timedelta(seconds=f.offset_s)
        print(f"  {utc_time}  offset={f.offset_s:.3f}s")
    total += len(p.frames)

print(f"\nTotal: {total} frames across {len(plans)} video(s)")

## 6. Parallel Extraction

For sessions with many video files, set `max_workers` to process multiple videos at once. Always pair this with `stream_output: true`; without it, each worker buffers an entire video in memory before writing and it will probably crash. The `stream_output` argument also writes frames out periodically instead of storing them all in memory until the run is completed, which can be helpful in large runs or on limited resources.

```yaml
# spec.yaml
rules:
  - interval_s: 10.0

max_workers: 4
stream_output: true

# Include {utc} when using multiple workers. Workers write to the same
# directory concurrently and {utc} guarantees no filename collisions.
filename_template: "{utc}_{video_stem}.jpg"
```

In [ ]:
from soi_frame_extractor import extract, spec_from_dict

spec = spec_from_dict({
    "rules": [{"interval_s": 10.0}],
    "max_workers":    4,
    "stream_output":  True,
    "filename_template": "{utc}_{video_stem}.jpg",
})

extract(spec=spec, video_source=video_dir, output_dir=output_dir)

frames = sorted(output_dir.glob("*.jpg"))
print(f"{len(frames)} frames written")

For more on `filename_template` arguments, refer to the Filenames and Metadata notebook.